# Laboratorio 2

## Integrantes

- Sergio Orellana 221122
- Ricardo Chuy 221007
- Rodrigo Mansilla 22611

## Task 3

### Implementen en Python Policy Iteration y Value Iteration para un MDP de ascensor simplificado

Acá implementamos los dos algoritmos para el MDP que fija el enunciado: edificio de 5 pisos, estado $(p,d)$ con el piso actual y un indicador de demanda pendiente en algún piso distinto, acciones subir, bajar y permanecer, transiciones deterministas, $\gamma=0.95$ y $\theta=10^{-6}$. Son 10 estados y 3 acciones, así que el MDP completo cabe en un par de tablas.

Antes de programar tuvimos que ponernos de acuerdo en tres cosas:

- El estado nos dice que hay demanda en algún piso distinto al actual, pero no en cuál. Decidimos entonces que cualquier movimiento que efectivamente cambie de piso llega a donde estaba la solicitud y la atiende, con lo que se cobra el $+10$ y el sistema pasa a $(p',0)$. Si el ascensor se queda quieto, o intenta moverse estando en un extremo del edificio, la demanda sigue pendiente y solo recibe el $-1$.
- El $-1$ lo aplicamos a todo paso en el que no se atiende a nadie, incluidos los pasos donde no hay demanda. Lo pensamos como el costo de tener el sistema andando sin haber resuelto nada en ese instante.
- El $-0.5$ lo cobramos cada vez que se da una orden de movimiento sin demanda, aunque el ascensor no cambie de piso por estar en el piso 1 o en el 5, porque el motor se activa igual y ese es justamente el consumo que se quiere evitar.

Algo que salta rápido es que al ser todo determinista no hay manera de que aparezcan solicitudes nuevas, así que una vez que el sistema llega a $d=0$ se queda ahí. Los estados sin demanda terminan siendo absorbentes y eso es lo que más lo aleja del MDP que modelamos en la Task 1.

In [1]:
# El enunciado prohíbe librerías de RL, así que el MDP y los dos algoritmos
# los construimos desde cero con numpy.
import time

import numpy as np

# Parámetros fijados por el enunciado.
GAMMA = 0.95      # factor de descuento
THETA = 1e-6      # umbral de convergencia

# Componentes de la recompensa r(s, a, s').
R_PASO = -1.0             # cada paso sin atender demanda
R_ATENDER = 10.0          # atender una demanda pendiente
R_MOVIMIENTO = -0.5       # movimiento innecesario en ausencia de demanda

print("Parámetros del MDP")
print()
print("gamma:", GAMMA)
print("theta:", THETA)
print("recompensas:", R_PASO, R_ATENDER, R_MOVIMIENTO)

Parámetros del MDP

gamma: 0.95
theta: 1e-06
recompensas: -1.0 10.0 -0.5


In [2]:
# Espacio de estados S = {(p, d)}, con p el piso y d el indicador de demanda.
PISOS = [1, 2, 3, 4, 5]
DEMANDA = [0, 1]

ESTADOS = [(p, d) for p in PISOS for d in DEMANDA]
INDICE = {estado: i for i, estado in enumerate(ESTADOS)}

# Espacio de acciones A. Usamos enteros para indexar las tablas y un
# diccionario aparte solo para imprimir resultados legibles.
SUBIR, BAJAR, PERMANECER = 0, 1, 2
ACCIONES = [SUBIR, BAJAR, PERMANECER]
NOMBRE_ACCION = {SUBIR: "subir", BAJAR: "bajar", PERMANECER: "permanecer"}

n_estados = len(ESTADOS)
n_acciones = len(ACCIONES)

print("Espacio de estados")
print()
for i, estado in enumerate(ESTADOS):
    print(f"índice {i}   estado (piso {estado[0]}, demanda {estado[1]})")

print()
print("Cantidad de estados:", n_estados)
print("Cantidad de acciones:", n_acciones)
print("Políticas deterministas posibles:", n_acciones ** n_estados)

Espacio de estados

índice 0   estado (piso 1, demanda 0)
índice 1   estado (piso 1, demanda 1)
índice 2   estado (piso 2, demanda 0)
índice 3   estado (piso 2, demanda 1)
índice 4   estado (piso 3, demanda 0)
índice 5   estado (piso 3, demanda 1)
índice 6   estado (piso 4, demanda 0)
índice 7   estado (piso 4, demanda 1)
índice 8   estado (piso 5, demanda 0)
índice 9   estado (piso 5, demanda 1)

Cantidad de estados: 10
Cantidad de acciones: 3
Políticas deterministas posibles: 59049


In [3]:
def paso(estado, accion):
    """Dinámica determinista del ascensor.

    Devuelve el estado siguiente s' y la recompensa r(s, a, s'). Al ser
    determinista, esta función define por completo p(s' | s, a): vale 1 para
    el sucesor que devuelve y 0 para todos los demás estados.
    """
    piso, demanda = estado

    # Topes del edificio: subir en el piso 5 y bajar en el piso 1 dejan
    # al ascensor donde estaba.
    if accion == SUBIR:
        piso_siguiente = min(piso + 1, 5)
    elif accion == BAJAR:
        piso_siguiente = max(piso - 1, 1)
    else:
        piso_siguiente = piso

    hubo_cambio_de_piso = piso_siguiente != piso

    # Con demanda pendiente, un movimiento efectivo llega al piso que la
    # solicitó, la atiende y deja el sistema sin demanda.
    if demanda == 1 and hubo_cambio_de_piso:
        return (piso_siguiente, 0), R_ATENDER

    # Con demanda pendiente, quedarse quieto o chocar contra un tope no atiende a nadie.
    if demanda == 1:
        return (piso, 1), R_PASO

    # Sin demanda, cualquier orden de movimiento suma el costo energético.
    recompensa = R_PASO
    if accion != PERMANECER:
        recompensa += R_MOVIMIENTO
    return (piso_siguiente, 0), recompensa

In [4]:
# Tablas explícitas del MDP, como pide el enunciado.
# P[s, a, s'] guarda p(s' | s, a) y R[s, a, s'] guarda r(s, a, s').
P = np.zeros((n_estados, n_acciones, n_estados))
R = np.zeros((n_estados, n_acciones, n_estados))

for s, estado in enumerate(ESTADOS):
    for a in ACCIONES:
        siguiente, recompensa = paso(estado, a)
        s_siguiente = INDICE[siguiente]
        P[s, a, s_siguiente] = 1.0          # transición determinista
        R[s, a, s_siguiente] = recompensa

print("Todas las distribuciones p(. | s, a) suman 1:", np.allclose(P.sum(axis=2), 1.0))
print()
print("Tabla de transiciones y recompensas")
print()

for s, estado in enumerate(ESTADOS):
    for a in ACCIONES:
        siguiente, recompensa = paso(estado, a)
        print(
            f"(piso {estado[0]}, demanda {estado[1]})"
            f"   acción {NOMBRE_ACCION[a]:11s}"
            f"   siguiente (piso {siguiente[0]}, demanda {siguiente[1]})"
            f"   recompensa {recompensa:5.1f}"
        )
    print()

Todas las distribuciones p(. | s, a) suman 1: True

Tabla de transiciones y recompensas

(piso 1, demanda 0)   acción subir         siguiente (piso 2, demanda 0)   recompensa  -1.5
(piso 1, demanda 0)   acción bajar         siguiente (piso 1, demanda 0)   recompensa  -1.5
(piso 1, demanda 0)   acción permanecer    siguiente (piso 1, demanda 0)   recompensa  -1.0

(piso 1, demanda 1)   acción subir         siguiente (piso 2, demanda 0)   recompensa  10.0
(piso 1, demanda 1)   acción bajar         siguiente (piso 1, demanda 1)   recompensa  -1.0
(piso 1, demanda 1)   acción permanecer    siguiente (piso 1, demanda 1)   recompensa  -1.0

(piso 2, demanda 0)   acción subir         siguiente (piso 3, demanda 0)   recompensa  -1.5
(piso 2, demanda 0)   acción bajar         siguiente (piso 1, demanda 0)   recompensa  -1.5
(piso 2, demanda 0)   acción permanecer    siguiente (piso 2, demanda 0)   recompensa  -1.0

(piso 2, demanda 1)   acción subir         siguiente (piso 3, demanda 0)   recom

In [5]:
def valor_q(s, a, V, gamma):
    """q(s, a) = suma_{s'} p(s' | s, a) [ r(s, a, s') + gamma * V(s') ].

    Es el bloque que aparece dentro de las dos ecuaciones de Bellman de las
    diapositivas. La de expectativa es V(s) = q(s, pi(s)) y la de optimalidad
    es V(s) = max_a q(s, a), así que las dos se arman con esta misma función.

    Aunque las transiciones son deterministas y la suma tiene un solo término
    distinto de cero, la escribimos como producto punto completo para que el
    código refleje la fórmula general.
    """
    return float(np.sum(P[s, a] * (R[s, a] + gamma * V)))


def politica_greedy(V, gamma):
    """Política voraz respecto de V: pi(s) = argmax_a q(s, a)."""
    return np.array(
        [int(np.argmax([valor_q(s, a, V, gamma) for a in ACCIONES])) for s in range(n_estados)]
    )

In [6]:
def policy_evaluation(politica, gamma=GAMMA, theta=THETA, V_inicial=None):
    """Policy Evaluation, en función separada para que Policy Iteration la reutilice.

    Aplica la ecuación de Bellman de expectativa

        V(s) = suma_{s'} p(s' | s, pi(s)) [ r(s, pi(s), s') + gamma * V(s') ]

    hasta que el cambio máximo entre barridos queda por debajo de theta. El
    operador es una contracción de factor gamma, así que converge a V^pi desde
    cualquier inicialización.

    Recibe un valor inicial opcional para arrancar desde la estimación del ciclo
    anterior en vez de reiniciar en ceros. Devuelve V, la cantidad de barridos
    completos sobre S, las aplicaciones del operador de Bellman (una por cada
    estado actualizado) y la cantidad de valores q calculados.
    """
    V = np.zeros(n_estados) if V_inicial is None else V_inicial.copy()

    barridos = 0
    aplicaciones = 0

    while True:
        delta = 0.0
        for s in range(n_estados):
            v_anterior = V[s]

            # V(s) <- q(s, pi(s)), la ecuación de Bellman de expectativa.
            V[s] = valor_q(s, politica[s], V, gamma)

            aplicaciones += 1
            delta = max(delta, abs(v_anterior - V[s]))

        barridos += 1

        # Criterio de paro: ||V_{k+1} - V_k||_inf < theta.
        if delta < theta:
            break

    # Acá cada aplicación calcula un solo q, el de la acción que dicta la política.
    return V, barridos, aplicaciones, aplicaciones

In [7]:
def policy_iteration(gamma=GAMMA, theta=THETA):
    """Policy Iteration completo: evaluación y mejora alternadas.

    En cada ciclo evalúa la política actual y después la mejora tomando
    pi(s) = argmax_a q(s, a). Por el teorema de mejora de políticas la nueva
    política es al menos tan buena en todo estado, y como solo hay 3^10
    políticas deterministas el proceso termina en una cantidad finita de ciclos.
    """
    inicio = time.perf_counter()

    # Arrancamos con el ascensor quieto en todos los estados.
    politica = np.full(n_estados, PERMANECER, dtype=int)
    V = np.zeros(n_estados)

    iteraciones_externas = 0
    barridos_totales = 0
    aplicaciones_bellman = 0
    evaluaciones_q = 0

    while True:
        # Fase 1, evaluación. Arranca desde la V del ciclo anterior.
        V, barridos, aplicaciones, q_usados = policy_evaluation(politica, gamma, theta, V)
        barridos_totales += barridos
        aplicaciones_bellman += aplicaciones
        evaluaciones_q += q_usados

        # Fase 2, mejora voraz respecto de V.
        politica_estable = True
        for s in range(n_estados):
            accion_anterior = politica[s]

            # pi(s) <- argmax_a q(s, a).
            politica[s] = int(np.argmax([valor_q(s, a, V, gamma) for a in ACCIONES]))

            aplicaciones_bellman += 1
            evaluaciones_q += n_acciones

            if politica[s] != accion_anterior:
                politica_estable = False

        iteraciones_externas += 1

        # Criterio de paro: la política ya no cambia en ningún estado.
        if politica_estable:
            break

    tiempo = time.perf_counter() - inicio

    return {
        "nombre": "Policy Iteration",
        "V": V,
        "politica": politica,
        "iteraciones": iteraciones_externas,
        "barridos": barridos_totales,
        "aplicaciones_bellman": aplicaciones_bellman,
        "evaluaciones_q": evaluaciones_q,
        "tiempo": tiempo,
    }

In [8]:
def value_iteration(gamma=GAMMA, theta=THETA):
    """Value Iteration.

    Aplica directamente la ecuación de Bellman de optimalidad

        V(s) = max_a suma_{s'} p(s' | s, a) [ r(s, a, s') + gamma * V(s') ]

    en cada barrido, sin evaluar ninguna política por completo. La política se
    extrae una sola vez al final, tomando la acción voraz respecto de V*.
    """
    inicio = time.perf_counter()

    V = np.zeros(n_estados)

    iteraciones = 0
    aplicaciones_bellman = 0
    evaluaciones_q = 0

    while True:
        delta = 0.0
        for s in range(n_estados):
            v_anterior = V[s]

            # V(s) <- max_a q(s, a). Cada barrido revisa las 3 acciones de cada
            # estado, por eso cuesta más que uno de Policy Evaluation.
            V[s] = max(valor_q(s, a, V, gamma) for a in ACCIONES)

            aplicaciones_bellman += 1
            evaluaciones_q += n_acciones
            delta = max(delta, abs(v_anterior - V[s]))

        iteraciones += 1

        # Criterio de paro: ||V_{k+1} - V_k||_inf < theta.
        if delta < theta:
            break

    politica = politica_greedy(V, gamma)
    evaluaciones_q += n_estados * n_acciones

    tiempo = time.perf_counter() - inicio

    return {
        "nombre": "Value Iteration",
        "V": V,
        "politica": politica,
        "iteraciones": iteraciones,
        "barridos": iteraciones,
        "aplicaciones_bellman": aplicaciones_bellman,
        "evaluaciones_q": evaluaciones_q,
        "tiempo": tiempo,
    }

In [9]:
def mostrar_resultado(resultado):
    """Tabla de valores V*(s), política óptima, iteraciones y tiempo de ejecución."""
    print(resultado["nombre"])
    print()
    print(f"{'estado':24s}{'V*(s)':>12s}   política óptima")

    for s, estado in enumerate(ESTADOS):
        etiqueta = f"(piso {estado[0]}, demanda {estado[1]})"
        accion = NOMBRE_ACCION[resultado["politica"][s]]
        print(f"{etiqueta:24s}{resultado['V'][s]:12.6f}   {accion}")

    print()
    print("Iteraciones hasta convergencia:", resultado["iteraciones"])
    print("Barridos completos sobre S:", resultado["barridos"])
    print("Aplicaciones del operador de Bellman:", resultado["aplicaciones_bellman"])
    print("Valores q calculados:", resultado["evaluaciones_q"])
    print(f"Tiempo de ejecución: {resultado['tiempo']:.6f} segundos")


resultado_pi = policy_iteration()
mostrar_resultado(resultado_pi)

Policy Iteration

estado                         V*(s)   política óptima
(piso 1, demanda 0)       -19.999983   permanecer
(piso 1, demanda 1)        -8.999983   subir
(piso 2, demanda 0)       -19.999983   permanecer
(piso 2, demanda 1)        -8.999983   subir
(piso 3, demanda 0)       -19.999983   permanecer
(piso 3, demanda 1)        -8.999983   subir
(piso 4, demanda 0)       -19.999983   permanecer
(piso 4, demanda 1)        -8.999983   subir
(piso 5, demanda 0)       -19.999983   permanecer
(piso 5, demanda 1)        -8.999984   bajar

Iteraciones hasta convergencia: 2
Barridos completos sobre S: 273
Aplicaciones del operador de Bellman: 2750
Valores q calculados: 2790
Tiempo de ejecución: 0.025862 segundos


In [10]:
resultado_vi = value_iteration()
mostrar_resultado(resultado_vi)

Value Iteration



estado                         V*(s)   política óptima
(piso 1, demanda 0)       -19.999982   permanecer
(piso 1, demanda 1)        -8.999982   subir
(piso 2, demanda 0)       -19.999982   permanecer
(piso 2, demanda 1)        -8.999982   subir
(piso 3, demanda 0)       -19.999982   permanecer
(piso 3, demanda 1)        -8.999982   subir
(piso 4, demanda 0)       -19.999982   permanecer
(piso 4, demanda 1)        -8.999982   subir
(piso 5, demanda 0)       -19.999982   permanecer
(piso 5, demanda 1)        -8.999983   bajar

Iteraciones hasta convergencia: 271
Barridos completos sobre S: 271
Aplicaciones del operador de Bellman: 2710
Valores q calculados: 8160
Tiempo de ejecución: 0.079523 segundos


In [11]:
# Comparación directa entre los dos algoritmos. Contamos también los valores q
# porque el barrido no cuesta lo mismo en uno que en otro.
politicas_iguales = np.array_equal(resultado_pi["politica"], resultado_vi["politica"])
diferencia_maxima = np.max(np.abs(resultado_pi["V"] - resultado_vi["V"]))

print("Comparación de los dos algoritmos")
print()
print(f"{'métrica':40s}{'Policy Iteration':>20s}{'Value Iteration':>20s}")
print(f"{'iteraciones hasta convergencia':40s}{resultado_pi['iteraciones']:>20d}{resultado_vi['iteraciones']:>20d}")
print(f"{'barridos completos sobre S':40s}{resultado_pi['barridos']:>20d}{resultado_vi['barridos']:>20d}")
print(f"{'aplicaciones del operador de Bellman':40s}{resultado_pi['aplicaciones_bellman']:>20d}{resultado_vi['aplicaciones_bellman']:>20d}")
print(f"{'valores q calculados':40s}{resultado_pi['evaluaciones_q']:>20d}{resultado_vi['evaluaciones_q']:>20d}")
print(f"{'tiempo en segundos':40s}{resultado_pi['tiempo']:>20.6f}{resultado_vi['tiempo']:>20.6f}")

print()
print("Las dos políticas óptimas coinciden en todos los estados:", politicas_iguales)
print("Diferencia máxima entre las dos tablas de valores:", diferencia_maxima)

Comparación de los dos algoritmos

métrica                                     Policy Iteration     Value Iteration
iteraciones hasta convergencia                             2                 271
barridos completos sobre S                               273                 271
aplicaciones del operador de Bellman                    2750                2710
valores q calculados                                    2790                8160
tiempo en segundos                                  0.025862            0.079523

Las dos políticas óptimas coinciden en todos los estados: True
Diferencia máxima entre las dos tablas de valores: 1.7911487120159109e-06


Los dos algoritmos llegan a la misma política y a valores que difieren en el orden de $10^{-6}$. Lo que dice esa política es bastante simple: si hay alguien esperando el ascensor se mueve, y si no hay demanda se queda quieto para no gastar energía. 